# Study 01: reproduce the frozen-data experiment

Upload Study01_Reproducibility.zip below. A standard CPU runtime is enough. This notebook reruns the recorded retrospective analysis, not current data. It does not access your Drive automatically or change sharing. Review README.md for limitations. Numerical code was tested in the hosted research runtime; this Colab interface has not been run end-to-end.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib, subprocess, sys

uploaded = files.upload()
name = next(n for n in uploaded if n.endswith(".zip"))
base = Path("/content/study01_reproduction").resolve()
base.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as archive:
    for member in archive.infolist():
        target = (base / member.filename).resolve()
        if not target.is_relative_to(base):
            raise ValueError("Unsafe archive path")
    archive.extractall(base)
study = base / "study01"
manifest = json.loads((study / "SHA256SUMS.json").read_text())
for relative, expected in manifest.items():
    assert hashlib.sha256((study / relative).read_bytes()).hexdigest() == expected, relative
print("Verified", len(manifest), "frozen files")


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(study / "requirements.txt")], check=True)


In [ ]:
for args in [["analyse.py", "--stage", "prepare"], ["analyse.py", "--stage", "validate"], ["analyse.py", "--stage", "final"], ["verify.py"], ["deliver.py"]]:
    subprocess.run([sys.executable, str(study / "code" / args[0]), *args[1:]], cwd=study, check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Image
results = pd.read_csv(study / "output/robustness_metrics.csv")
display(results[results.variant == "primary"][["model", "n", "MAE_pp", "RMSE_pp", "recall", "false_positive_rate"]])
for chart in ["chart_01_history.png", "chart_02_forecast_accuracy.png", "chart_03_robustness.png"]:
    display(Image(filename=str(study / "output" / chart)))


Expected primary MAE: A 1.6890898915; B 1.8724573516; C 1.7893909438. Small floating-point differences are possible. Re-running does not make the original historical test untouched again. Do not retune and claim the same test as new confirmation.
